# GraphRAG-Bench (ICLR 2026)

**Paper:** [When to use Graphs in RAG](https://arxiv.org/abs/2506.05690) — Xiang et al.  
**Resources:** [GitHub](https://github.com/GraphRAG-Bench/GraphRAG-Benchmark) · [HF dataset](https://huggingface.co/datasets/GraphRAG-Bench/GraphRAG-Bench) · [Leaderboard](https://graphrag-bench.github.io/)

Central question: *when do graph structures help vs vanilla vector RAG?*

| Level | Task | Paper observation |
|-------|------|-------------------|
| 1 | **Fact Retrieval** | Basic RAG ≈ / beats GraphRAG (Obs.1) |
| 2 | **Complex Reasoning** | GraphRAG wins (Obs.2) |
| 3 | **Contextual Summarize** | GraphRAG wins |
| 4 | **Creative Generation** | GraphRAG wins |

This notebook demos a **Novel-split mini subset** (Samuel Pepys diary book `Novel-4128`, ~2 Qs × 4 levels), runs local methods, maps question IDs → text, and shows the metric autopsy.


In [ ]:
# ── Config ─────────────────────────────────────────────────
N_PER_TYPE = 2
SOURCE = "Novel-4128"       # Pepys diary novel in GraphRAG-Bench
RUN_BENCHMARK = False       # True to re-run; False loads results_graphrag_bench/
REUSE_INDEXES = True
METHODS = [
    "semantic_rag",         # basic RAG — should compete on Fact Retrieval
    "rerank_semantic",
    "hybrid_rag",           # vec + GraphRAG local
    "frontier_rag",         # adaptive + CRAG escalate
]
# ───────────────────────────────────────────────────────────

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

from rag_benchmark import (
    BenchmarkConfig,
    BenchmarkRunner,
    build_graphrag_bench_subset,
    create_tracked_client,
)
from rag_benchmark.charts import METHOD_LABELS, plot_dashboard, print_leaderboard
from rag_benchmark.decision_playbook import build_decision_artifacts
from rag_benchmark.engineering import (
    build_engineering_scorecard,
    print_engineering_briefing,
    save_engineering_scorecard,
)

RESULTS = PROJECT_ROOT / "results_graphrag_bench"
RESULTS.mkdir(parents=True, exist_ok=True)
print(f"Project: {PROJECT_ROOT}")
print(f"Results: {RESULTS}")


## 1. Build GraphRAG-Bench Novel subset

Downloads the Novel corpus/questions from the official repo (cached under `data/graphrag_bench_cache/`), picks one book, filters out misaligned QA pairs, and writes:
- `data/corpus_graphrag_bench/*.txt`
- `data/qa/graphrag_bench_eval.json`
- `results/graphrag_bench_question_catalog.csv`


In [ ]:
built = build_graphrag_bench_subset(
    project_root=PROJECT_ROOT,
    n_per_type=N_PER_TYPE,
    source=SOURCE,
)
meta = built["meta"]
display(Markdown("### Subset meta"))
display(pd.Series({k: v for k, v in meta.items() if k != "findings"}).to_frame("value"))
display(Markdown(
    f"**Obs.1** {meta['findings']['obs1']}  \n"
    f"**Obs.2** {meta['findings']['obs2']}"
))

config = BenchmarkConfig.from_yaml(PROJECT_ROOT)
config.project_root = PROJECT_ROOT
config.corpus_dir = built["corpus_dir"]
config.qa_path = built["qa_path"]
config.semantic_collection = "graphrag_bench_semantic"
config.graph_workspace = PROJECT_ROOT / "graphrag_workspaces" / "graphrag_bench"
config.lazy_workspace = PROJECT_ROOT / "graphrag_workspaces" / "graphrag_bench_lazy"
config.reuse_indexes = REUSE_INDEXES
config.graph_indexing_method = "fast"
config.semantic_top_k = 8
config.max_documents = 10_000
config.results_dir = lambda: RESULTS  # type: ignore[method-assign]

print(f"Corpus docs: {len(list(Path(built['corpus_dir']).glob('*.txt')))}")
print(f"QA path: {built['qa_path']}")


## Demo: question ID → actual question

GraphRAG-Bench IDs look like `Novel-95217448`. Use this catalog in demos so the audience sees the **text**, not just hex.


In [ ]:
qa = json.loads(Path(built["qa_path"]).read_text(encoding="utf-8"))
catalog = pd.read_csv(PROJECT_ROOT / "results" / "graphrag_bench_question_catalog.csv")

# Enrich with live metric averages if a prior run exists
acc_path = RESULTS / "accuracy_results.csv"
if acc_path.exists():
    acc = pd.read_csv(acc_path)
    rows = []
    for _, row in catalog.iterrows():
        sub = acc[acc.question_id == row.question_id]
        rows.append({
            **row.to_dict(),
            "avg_judge": round(float(sub.llm_judge_score.mean()), 3) if len(sub) else None,
            "avg_f1": round(float(sub.token_f1.mean()), 3) if len(sub) else None,
            "em_rate": round(float(sub.exact_match.mean()), 3) if len(sub) else None,
            "contains_rate": round(float(sub.contains_answer.mean()), 3) if len(sub) else None,
            "best_method": (
                METHOD_LABELS.get(sub.loc[sub.composite_score.idxmax(), "method"], "")
                if len(sub) else ""
            ),
        })
    demo = pd.DataFrame(rows)
else:
    demo = catalog.copy()

display(Markdown("### Question catalog"))
display(demo)

print("\n── Full questions ──")
for i, q in enumerate(qa):
    print(f"Q{i+1} [{q['graphrag_bench_type']}]  {q['id']}")
    print(f"  Q:    {q['question']}")
    gold = str(q["expected_answer"])
    print(f"  Gold: {gold[:220]}{'…' if len(gold) > 220 else ''}\n")


## 2. Run methods (or load prior results)

Set `RUN_BENCHMARK = True` above to index + query. First hybrid/GraphRAG index on this novel can take a while; later runs reuse workspaces.


In [ ]:
results = []
if RUN_BENCHMARK:
    runner = BenchmarkRunner(config, create_tracked_client(config))
    results = runner.run_all(methods=METHODS)

    accuracy_df = BenchmarkRunner.to_accuracy_frame(results)
    qa_by_id = {q["id"]: q for q in qa}
    accuracy_df["graphrag_bench_type"] = accuracy_df["question_id"].map(
        lambda i: qa_by_id.get(i, {}).get("graphrag_bench_type", "")
    )
    summary_df = BenchmarkRunner.to_summary_frame(results, config)
    scenario_df = BenchmarkRunner.to_scenario_frame(results)
    token_df = BenchmarkRunner.to_token_frame(results, config)
    latency_df = BenchmarkRunner.to_latency_frame(results)

    accuracy_df.to_csv(RESULTS / "accuracy_results.csv", index=False)
    summary_df.to_csv(RESULTS / "summary.csv", index=False)
    scenario_df.to_csv(RESULTS / "scenario_results.csv", index=False)
    token_df.to_csv(RESULTS / "token_results.csv", index=False)
    latency_df.to_csv(RESULTS / "latency_results.csv", index=False)

    by_type = (
        accuracy_df.groupby(["graphrag_bench_type", "method"])["composite_score"]
        .mean()
        .reset_index()
        .sort_values(["graphrag_bench_type", "composite_score"], ascending=[True, False])
    )
    by_type.to_csv(RESULTS / "by_question_type.csv", index=False)

    plot_dashboard(RESULTS)
    scorecard = build_engineering_scorecard(summary_df, scenario_df, accuracy_df)
    save_engineering_scorecard(scorecard, RESULTS)
    build_decision_artifacts(RESULTS, Path(built["qa_path"]))
    print_engineering_briefing(scorecard)
    print(by_type.to_string(index=False))
else:
    assert (RESULTS / "accuracy_results.csv").exists(), (
        "No results_graphrag_bench/ yet — set RUN_BENCHMARK=True or run "
        "`PYTHONPATH=src python scripts/run_graphrag_bench.py`"
    )
    print(f"Loaded prior results from {RESULTS}")
    print(pd.read_csv(RESULTS / "summary.csv")[
        ["method", "mean_composite_score", "mean_llm_judge", "tokens_per_query", "mean_query_latency_seconds"]
    ].round(3).to_string(index=False))


## 3. When do graphs help? (by GraphRAG-Bench level)

Compare winners to the paper: vector should stay competitive on **Fact Retrieval**; hybrid/graph paths should pull ahead on harder levels.


In [ ]:
acc = pd.read_csv(RESULTS / "accuracy_results.csv")
if "graphrag_bench_type" not in acc.columns:
    qa_by_id = {q["id"]: q for q in json.loads(Path(built["qa_path"]).read_text())}
    acc["graphrag_bench_type"] = acc["question_id"].map(
        lambda i: qa_by_id.get(i, {}).get("graphrag_bench_type", "")
    )

# Drop rows whose question_id is not in the current subset (stale runs)
valid_ids = {q["id"] for q in qa}
stale = ~acc.question_id.isin(valid_ids)
if stale.any():
    display(Markdown(
        f"**Note:** {int(stale.sum())} result rows are from an older subset — "
        f"re-run with `RUN_BENCHMARK=True` for a clean leaderboard."
    ))

by_type = (
    acc.groupby(["graphrag_bench_type", "method"])[[
        "composite_score", "llm_judge_score", "token_f1", "exact_match", "contains_answer"
    ]]
    .mean()
    .reset_index()
)
by_type["label"] = by_type["method"].map(lambda m: METHOD_LABELS.get(m, m))

order = [
    "Fact Retrieval",
    "Complex Reasoning",
    "Contextual Summarize",
    "Creative Generation",
]
winners = []
for qt in order:
    sub = by_type[by_type.graphrag_bench_type == qt]
    if sub.empty:
        continue
    best = sub.loc[sub.composite_score.idxmax()]
    winners.append({
        "task_level": qt,
        "winner": best["label"],
        "composite": round(float(best.composite_score), 3),
        "judge": round(float(best.llm_judge_score), 3),
        "contains": round(float(best.contains_answer), 3),
        "em": round(float(best.exact_match), 3),
        "paper_hint": "vector OK" if qt == "Fact Retrieval" else "graphs help",
    })

display(Markdown("### Winners by task level"))
display(pd.DataFrame(winners))

pivot = by_type.pivot_table(
    index="label", columns="graphrag_bench_type", values="composite_score"
)
pivot = pivot.reindex(columns=[c for c in order if c in pivot.columns])
display(Markdown("### Composite by method × task level"))
display(pivot.round(3))

# Bar chart of winners' composites
if winners:
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.barh(
        [w["task_level"] for w in winners][::-1],
        [w["composite"] for w in winners][::-1],
        color="#3b6d9a",
    )
    ax.set_xlabel("Best-method composite (0–1)")
    ax.set_title("GraphRAG-Bench mini — best composite per task level")
    for y, w in enumerate(winners[::-1]):
        ax.text(w["composite"] + 0.01, y, w["winner"], va="center", fontsize=9)
    plt.tight_layout()
    plt.show()


## 4. Metric autopsy (why EM looks “broken”)

GraphRAG-Bench gold answers are often **sentences**, not Hotpot-style spans. Exact match stays near zero; **LLM judge** and **contains** are the readable signals for demos.


In [ ]:
metric_means = (
    acc.groupby("method")[["llm_judge_score", "contains_answer", "token_f1", "exact_match"]]
    .mean()
    .rename(index=lambda m: METHOD_LABELS.get(m, m))
    .sort_values("llm_judge_score")
)
display(Markdown("### Mean metrics by method"))
display(metric_means.round(3))

fig, ax = plt.subplots(figsize=(10, 4.5))
metric_means.plot(kind="barh", ax=ax)
ax.set_xlim(0, 1)
ax.set_xlabel("Mean score (0–1)")
ax.set_title("GraphRAG-Bench subset — judge vs lexical metrics")
plt.tight_layout()
plt.show()

n = len(acc)
hi = int(((acc.llm_judge_score >= 0.5) & (~acc.exact_match.astype(bool))).sum())
display(Markdown(
    f"- **{hi}/{n}** rows: judge ≥ 0.5 but EM = 0\n"
    f"- Prefer **judge + contains** for generative GraphRAG demos; EM/F1 for extractive spans only."
))


## 5. Engineering scorecard + charts

Tokens, latency, and routing — what you ship, not just a single average.


In [ ]:
print_leaderboard(RESULTS)
brief = RESULTS / "engineering_briefing.md"
if brief.exists():
    display(Markdown(brief.read_text(encoding="utf-8")))
dash = RESULTS / "benchmark_dashboard.png"
if dash.exists():
    display(Image(filename=str(dash)))
cheat = RESULTS / "routing_cheatsheet.md"
if cheat.exists():
    display(Markdown("### Routing cheatsheet (truncated)"))
    display(Markdown(cheat.read_text(encoding="utf-8")[:3000]))


## 6. Full GraphRAG-Bench vs this harness

| Paper component | This demo |
|-----------------|-----------|
| Novel + Medical corpora | **Novel** mini-subset (one book, Pepys) |
| 4 task levels | Same labels; ~2 Qs each |
| Indexing / retrieval / generation evals | Generation (judge+EM+F1+contains) + tokens/latency |
| Official `Evaluation/` (ROUGE, coverage, faithfulness) | Next upgrade — wire their scripts for leaderboard parity |
| Public leaderboard | [graphrag-bench.github.io](https://graphrag-bench.github.io) |

**CLI**

```bash
PYTHONPATH=src python scripts/run_graphrag_bench.py semantic_rag,rerank_semantic,hybrid_rag,frontier_rag 2 Novel-4128
```

**Citation:** Xiang et al., *When to use Graphs in RAG*, ICLR 2026 — arXiv:2506.05690
